# NASA C-MAPSS FD001 degradation-stage analysis (Kaggle)

Run this only after downloading `cmapss_fd001_forecasting_results.zip`. It groups fixed held-out forecast errors by the predeclared normalized-life bins; it does not fit a model or predict RUL.

In [ ]:
!pip -q uninstall -y jax jaxlib
!pip -q install tensorflow==2.18.0 keras==3.15.1 numpy==2.0.2 pandas==2.2.2 scikit-learn==1.9.0
!python -c "import sys,tensorflow as tf,sklearn; print({'python':sys.version.split()[0], 'tensorflow':tf.__version__, 'sklearn':sklearn.__version__, 'devices':[str(x) for x in tf.config.list_physical_devices()]})"

In [ ]:
from pathlib import Path
import hashlib, json, shutil, zipfile
import numpy as np
INPUT = Path('/kaggle/input'); bundles = list(INPUT.rglob('kaggle_cmapss_stage_analysis_input_bundle.zip')); forecast_archives = list(INPUT.rglob('cmapss_fd001_forecasting_results.zip'))
assert len(bundles) == 1, f'Attach exactly one Dataset containing kaggle_cmapss_stage_analysis_input_bundle.zip; found {bundles}'
assert len(forecast_archives) == 1, f'Attach exactly one forecasting output ZIP named cmapss_fd001_forecasting_results.zip; found {forecast_archives}'
UNPACKED = Path('/kaggle/working/cmapss_stage_input')
if UNPACKED.exists(): shutil.rmtree(UNPACKED)
shutil.unpack_archive(bundles[0], UNPACKED)
candidates = [p for p in UNPACKED.rglob('cmapss_stage_analysis.py') if p.parent.name == 'ieee_experiments' and p.parent.parent.name == 'research_v2']
assert len(candidates) == 1, f'Bundle must contain exactly one stage script; found {candidates}'
SOURCE_ROOT = candidates[0].parents[2]; WORK = Path('/kaggle/working/autosense_cmapss_stage')
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(SOURCE_ROOT, WORK)
expected = {'research_v2/ieee_experiments/cmapss_stage_analysis.py':'ffe4ae498ce79d18d22e619f4fb498bfcca4d229070949e341e788ddd093d691','research_v2/CODEX_IEEE_EXECUTION_PLAN.md':'1e72e4109507e7968af365367d855fac495439173f38c4c7a72c4ece71c627bb','research_v2/external_data/cmapss/train_FD001.txt':'963b5e22825b34d8b21c69e1aeb4af3e647050eb672ee8834ba4b5d91d2de0f8','research_v2/ieee_results/cmapss_fd001_split.json':'9e4c0a4ab7cef63169c1bc87a52885af983ffab5ea8f0e8e886d352e63779003'}
actual = {rel:hashlib.sha256((WORK/rel).read_bytes()).hexdigest() for rel in expected}; assert actual == expected, json.dumps({'expected':expected,'actual':actual},indent=2)
with zipfile.ZipFile(forecast_archives[0]) as z: z.extractall(WORK / 'research_v2/ieee_results')
predictions = sorted((WORK/'research_v2/ieee_results/cmapss_fd001_predictions').glob('*.npz')); assert len(predictions) == 44, f'Forecast ZIP must contain 44 prediction artifacts, got {len(predictions)}'
for p in predictions:
    saved=np.load(p,allow_pickle=False); assert saved['dataset_sha256'].item() == expected['research_v2/external_data/cmapss/train_FD001.txt']; assert saved['experiment_script_sha256'].item() == 'cee828114ca0e22800b6bf12faa9643db4646f0c764bbef81558e1a6243a3a65'
print(f'Validated stage inputs and all 44 exact forecasting prediction artifacts; writable root: {WORK}')

In [ ]:
%cd /kaggle/working/autosense_cmapss_stage
!python -u research_v2/ieee_experiments/cmapss_stage_analysis.py

In [ ]:
import json, platform, sys, zipfile
import pandas as pd, tensorflow as tf
OUT=WORK/'research_v2/ieee_results'; result=OUT/'cmapss_stage_metrics.csv'; assert result.exists()
df=pd.read_csv(result); required={'Dataset','Configuration','Target sensor','Model','Seed','Stage','Forecast origins','MAE','RMSE','R2','Unit','Stage definition','RUL use','Prediction dataset SHA256','Forecasting script SHA256','Stage analysis script SHA256','Protocol SHA256','Model parameters','Epochs run','Best epoch','Training seconds'}
assert required <= set(df.columns), sorted(required-set(df.columns)); assert len(df)==132 and not df.duplicated(['Configuration','Target sensor','Model','Seed','Stage']).any()
assert set(df['Stage'])=={'early (<0.5 normalized cycle fraction)','mid (0.5 to <0.8 normalized cycle fraction)','late (>=0.8 normalized cycle fraction)'}
assert df['Prediction dataset SHA256'].eq(expected['research_v2/external_data/cmapss/train_FD001.txt']).all(); assert df['Forecasting script SHA256'].eq('cee828114ca0e22800b6bf12faa9643db4646f0c764bbef81558e1a6243a3a65').all()
manifest={'phase':'cmapss_stage_analysis','input_sha256':actual,'forecast_zip':str(forecast_archives[0]),'python':sys.version,'tensorflow':tf.__version__,'platform':platform.platform(),'devices':[str(x) for x in tf.config.list_physical_devices()],'expected_rows':132,'validation':'44 fixed forecast artifacts x 3 predeclared stages; no duplicate configuration/sensor/model/seed/stage rows'}
manifest_path=OUT/'kaggle_cmapss_stage_analysis_run_manifest.json'; manifest_path.write_text(json.dumps(manifest,indent=2))
zip_path=Path('/kaggle/working/cmapss_stage_analysis_results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z: z.write(result,result.name); z.write(manifest_path,manifest_path.name)
print(df.groupby(['Stage','Model'])[['MAE','RMSE','R2']].mean().round(6).to_string()); print(f'Validated 132 stage rows. Download: {zip_path}')